# 🇻🇳 Extractive Summarization — Vietnamese
### PhoBERT + LoRA Fine-tuning trên OpenHust/vietnamese-summarization

## 1. Cài đặt thư viện

In [ ]:
!pip install -q transformers datasets evaluate rouge_score rouge-score \
    sentencepiece accelerate huggingface_hub peft bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/NLP_Projects/phobert-ext-summ"
os.makedirs(save_dir, exist_ok=True)

Mounted at /content/drive


In [ ]:
import os
import gc
import re
import json
import logging
import warnings
from typing import List, Dict, Tuple, Optional
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset as TorchDataset

from datasets import Dataset, DatasetDict, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel,
)
from sklearn.metrics import f1_score, precision_score, recall_score
import evaluate

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

SEED = 42
set_seed(SEED)

## 2. Cấu hình (Tối ưu cho Kaggle T4)

In [ ]:
CONFIG = {
    # ─── Model ───
    "model_name": "vinai/phobert-base-v2",

    # ─── LoRA ───
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.1,
    "lora_target_modules": ["query", "value"],

    # ─── Dataset ───
    "dataset_name": "OpenHust/vietnamese-summarization",
    "text_column": "Document",
    "summary_column": "Summary",

    # ─── Extractive ───
    "max_sentences": 25,
    "max_select_sentences": 3,

    # ─── Tokenizer ───
    "max_length": 256,

    "output_dir": "/content/drive/MyDrive/NLP_Projects/phobert-ext-summ",

    "num_train_epochs": 3,

    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "gradient_accumulation_steps": 2,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "warmup_steps": 100,
    "lr_scheduler_type": "cosine",
    "fp16": True,
    "gradient_checkpointing": True,
    "optim": "adamw_torch_fused",

    # ─── Eval & Save & Early Stopping ───
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "early_stopping_patience": 1,

    # ─── Inference ───
    "top_k_sentences": 3,

    # ─── Split ───
    "test_size": 0.1,
    "val_size": 0.1,
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)

## 3. Load Dataset OpenHust

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

def load_openhust_summarization(dataset_name: str) -> Dataset:
    try:
        ds = load_dataset(dataset_name, split="train")
        return ds
    except Exception as e:
        pass
    api = HfApi()
    files = api.list_repo_files(dataset_name, repo_type="dataset")
    csv_files = [f for f in files if f.endswith(".csv")]
    print(f"   Tìm thấy {len(csv_files)} files")

    all_dfs = []
    for csv_file in csv_files:
        try:
            path = hf_hub_download(repo_id=dataset_name, filename=csv_file, repo_type="dataset")
            df = pd.read_csv(path)
            df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], errors="ignore")
            if {"Document", "Summary"}.issubset(df.columns):
                all_dfs.append(df)
                print(f"   ✓ {csv_file}: {len(df):,}")
        except Exception as e:
            print(f"   ✗ {csv_file}: {e}")

    merged = pd.concat(all_dfs, ignore_index=True)
    merged = merged.dropna(subset=["Document", "Summary"])
    merged = merged[
        (merged["Document"].str.strip().str.len() > 10) &
        (merged["Summary"].str.strip().str.len() > 5)
    ].reset_index(drop=True)

    print(f"\n Tổng: {len(merged):,} samples")
    return Dataset.from_pandas(merged)


raw_dataset = load_openhust_summarization(CONFIG["dataset_name"])
print(f"   Columns: {raw_dataset.column_names}")


README.md: 0.00B [00:00, ?B/s]

Kmeans_1024_new.csv:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

Kmeans_512_new.csv:   0%|          | 0.00/25.0M [00:00<?, ?B/s]

Kmeans_512_token_new.csv:   0%|          | 0.00/29.1M [00:00<?, ?B/s]

bio_medicine.csv:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

herding_512_bio_medicine.csv:   0%|          | 0.00/25.1M [00:00<?, ?B/s]

herding_bio_medicine.csv:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

herding_prompt_512_bio_medicine.csv:   0%|          | 0.00/29.1M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

   Columns: ['Unnamed: 0', 'Document', 'Summary', 'Dataset']


## 4. Tạo Extractive Labels (Greedy Oracle)

Mỗi câu trong document được gán label:
- `1` = câu quan trọng (nên đưa vào tóm tắt)
- `0` = câu không quan trọng

Label được tạo bằng **Greedy Oracle**: lần lượt chọn câu nào giúp tăng ROUGE nhiều nhất.

In [ ]:
_ABBRS = [
    "TP.", "PGS.", "TS.", "GS.", "ThS.", "BS.", "KS.", "CN.",
    "Mr.", "Mrs.", "Ms.", "Dr.", "Prof.", "tr.", "tg.", "nxb.", "Nxb.",
]
_ABBR_MAP = {f"__A{i}__": a for i, a in enumerate(_ABBRS)}


def split_sentences_vi(text: str) -> List[str]:
    text = text.strip()
    if not text:
        return []

    for placeholder, abbr in _ABBR_MAP.items():
        text = text.replace(abbr, placeholder)

    sents = re.split(
        r'(?<=[.!?])\s+(?=[A-ZÀÁẢÃẠĂẮẰẲẴẶÂẤẦẨẪẬÈÉẺẼẸÊẾỀỂỄỆÌÍỈĨỊÒÓỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢÙÚỦŨỤƯỨỪỬỮỰỲÝỶỸỴĐ"\(])',
        text
    )

    if len(sents) <= 1:
        sents = re.split(r'(?<=[.!?])\s+', text)

    result = []
    for s in sents:
        for placeholder, abbr in _ABBR_MAP.items():
            s = s.replace(placeholder, abbr)
        s = s.strip()
        if len(s) > 5:
            result.append(s)
    return result


test_sents = split_sentences_vi(raw_dataset[0]["Document"])
print(f"Test: {len(test_sents)} câu")
for i, s in enumerate(test_sents[:3]):
    print(f"  [{i}] {s[:100]}{'...' if len(s)>100 else ''}")

Test: 5 câu
  [0] Đây là một trong những nội dung tại văn bản vừa được UBND TP Hà Nội ban hành về việc tăng cường công...
  [1] , một chủ sạp bán thịt chó sống ở đây , cho hay trong 10 ngày đầu mỗi tháng âm lịch , các cửa hàng v...
  [2] , trung bình mỗi tháng chị bán được khoảng 3 tạ thịt chó sống .Tương tự , tại phố Nguyễn Khang ( quậ...


In [ ]:
from rouge_score import rouge_scorer

_scorer = rouge_scorer.RougeScorer(["rouge2"], use_stemmer=False)


def _rouge2_f1(hyp: str, ref: str) -> float:
    if not hyp or not ref:
        return 0.0
    return _scorer.score(ref, hyp)["rouge2"].fmeasure


def greedy_oracle(sentences: List[str], reference: str, max_select: int = 3) -> List[int]:
    n = len(sentences)
    if n == 0:
        return []

    labels = [0] * n
    selected = []
    current = ""
    current_score = 0.0

    for _ in range(min(max_select, n)):
        best_idx, best_score = -1, current_score

        for i in range(n):
            if i in selected:
                continue
            candidate = (current + " " + sentences[i]).strip()
            score = _rouge2_f1(candidate, reference)
            if score > best_score:
                best_score = score
                best_idx = i

        if best_idx >= 0:
            selected.append(best_idx)
            current = (current + " " + sentences[best_idx]).strip()
            current_score = best_score
        else:
            break

    for idx in selected:
        labels[idx] = 1
    return labels


sents = split_sentences_vi(raw_dataset[0]["Document"])
ref = raw_dataset[0]["Summary"]
labs = greedy_oracle(sents, ref)
print(f"Oracle chọn {sum(labs)}/{len(labs)} câu")
for i, (s, l) in enumerate(zip(sents, labs)):
    if l == 1:
        print(f"  [{i}] {s[:120]}")

Oracle chọn 1/5 câu
  [0] Đây là một trong những nội dung tại văn bản vừa được UBND TP Hà Nội ban hành về việc tăng cường công tác quản lý nuôi , 


In [ ]:
from tqdm.auto import tqdm

def build_extractive_dataset(dataset: Dataset) -> pd.DataFrame:
    """Chuyển document-level → sentence-level dataset."""
    rows = []
    oracle_scores = []
    max_sents = CONFIG["max_sentences"]
    max_sel = CONFIG["max_select_sentences"]

    for doc_id in tqdm(range(len(dataset)), desc="Building labels"):
        doc = str(dataset[doc_id][CONFIG["text_column"]])
        ref = str(dataset[doc_id][CONFIG["summary_column"]])

        sents = split_sentences_vi(doc)[:max_sents]
        if not sents:
            continue

        labels = greedy_oracle(sents, ref, max_select=max_sel)

        oracle_summ = " ".join(s for s, l in zip(sents, labels) if l == 1)
        if oracle_summ:
            oracle_scores.append(_rouge2_f1(oracle_summ, ref))

        for idx, (sent, label) in enumerate(zip(sents, labels)):
            rows.append({
                "doc_id": doc_id,
                "sent_idx": idx,
                "sentence": sent,
                "label": label,
            })

    df = pd.DataFrame(rows)
    print(f"\n Extractive Dataset:")
    print(f"   Sentences: {len(df):,}")
    print(f"   Documents: {df['doc_id'].nunique():,}")
    print(f"   Positive ratio: {df['label'].mean():.1%}")
    print(f"   Oracle ROUGE-2: {np.mean(oracle_scores):.4f} (avg)")
    return df

subset_dataset = raw_dataset.select(range(18000))
ext_df = build_extractive_dataset(subset_dataset)

Building labels:   0%|          | 0/18000 [00:00<?, ?it/s]


 Extractive Dataset:
   Sentences: 175,683
   Documents: 18,000
   Positive ratio: 17.2%
   Oracle ROUGE-2: 0.2625 (avg)


In [ ]:
from sklearn.model_selection import train_test_split

all_doc_ids = ext_df["doc_id"].unique()

train_val_ids, test_ids = train_test_split(all_doc_ids, test_size=0.1, random_state=SEED)
train_ids, val_ids = train_test_split(train_val_ids, test_size=0.1111, random_state=SEED)

train_ids_subset = train_ids[:15000]
val_ids_subset   = val_ids[:1500]
test_ids_subset  = test_ids[:1500]

train_df = ext_df[ext_df["doc_id"].isin(train_ids_subset)].reset_index(drop=True)
val_df   = ext_df[ext_df["doc_id"].isin(val_ids_subset)].reset_index(drop=True)
test_df  = ext_df[ext_df["doc_id"].isin(test_ids_subset)].reset_index(drop=True)

print(f"   Train: {len(train_df):,} sents ({train_df['doc_id'].nunique():,} docs) — pos: {train_df['label'].mean():.1%}")
print(f"   Val:   {len(val_df):,} sents ({val_df['doc_id'].nunique():,} docs) — pos: {val_df['label'].mean():.1%}")
print(f"   Test:  {len(test_df):,} sents ({test_df['doc_id'].nunique():,} docs) — pos: {test_df['label'].mean():.1%}")


   Train: 140,572 sents (14,400 docs) — pos: 17.2%
   Val:   14,375 sents (1,500 docs) — pos: 17.4%
   Test:  15,006 sents (1,500 docs) — pos: 16.7%


## 5. Model: PhoBERT + LoRA + Classification Head

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])


def tokenize_fn(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        max_length=CONFIG["max_length"],
    )

train_hf = Dataset.from_pandas(train_df[["sentence", "label"]])
val_hf   = Dataset.from_pandas(val_df[["sentence", "label"]])
test_hf  = Dataset.from_pandas(test_df[["sentence", "label"]])

print("Tokenizing...")
train_tok = train_hf.map(tokenize_fn, batched=True, batch_size=2000, remove_columns=["sentence"])
val_tok   = val_hf.map(tokenize_fn, batched=True, batch_size=2000, remove_columns=["sentence"])
test_tok  = test_hf.map(tokenize_fn, batched=True, batch_size=2000, remove_columns=["sentence"])

print(f"Tokenized — Train: {len(train_tok):,}, Val: {len(val_tok):,}, Test: {len(test_tok):,}")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing...


Map:   0%|          | 0/140572 [00:00<?, ? examples/s]

Map:   0%|          | 0/14375 [00:00<?, ? examples/s]

Map:   0%|          | 0/15006 [00:00<?, ? examples/s]

Tokenized — Train: 140,572, Val: 14,375, Test: 15,006


In [ ]:

print(f"Loading {CONFIG['model_name']}...")

base_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=2,
    problem_type="single_label_classification",
)

if CONFIG["gradient_checkpointing"]:
    base_model.gradient_checkpointing_enable()
    if hasattr(base_model, "enable_input_require_grads"):
        base_model.enable_input_require_grads()
    else:
        def make_inputs_require_grad(module, input, output):
            output.requires_grad_(True)
        base_model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["lora_target_modules"],
    bias="none",
    modules_to_save=["classifier"],
)

model = get_peft_model(base_model, lora_config)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n Model Info:")
print(f"   Total params:     {total_params:,} ({total_params/1e6:.1f}M)")
print(f"   Trainable params: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
print(f"   Trainable ratio:  {trainable_params/total_params:.1%}")
print(f"   Gradient Checkpointing: {CONFIG['gradient_checkpointing']}")


Loading vinai/phobert-base-v2...


pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]


 Model Info:
   Total params:     136,181,764 (136.2M)
   Trainable params: 1,181,954 (1.18M)
   Trainable ratio:  0.9%
   Gradient Checkpointing: True


## 6. Training với Weighted Loss

In [ ]:
pos_ratio = train_df["label"].mean()
w0 = 1.0 / (2 * (1 - pos_ratio))
w1 = 1.0 / (2 * pos_ratio)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = torch.tensor([w0, w1], dtype=torch.float32).to(device)
print(f"Class weights: [label=0: {w0:.3f}, label=1: {w1:.3f}]")
print(f"(Pos ratio = {pos_ratio:.1%}, weight ratio = {w1/w0:.1f}x)")


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = nn.CrossEntropyLoss(weight=class_weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1": f1_score(labels, preds, pos_label=1),
        "precision": precision_score(labels, preds, pos_label=1, zero_division=0),
        "recall": recall_score(labels, preds, pos_label=1, zero_division=0),
        "accuracy": (preds == labels).mean(),
    }

Class weights: [label=0: 0.604, label=1: 2.904]
(Pos ratio = 17.2%, weight ratio = 4.8x)


In [ ]:
model.config.use_cache = False

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_steps=CONFIG["warmup_steps"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    fp16=CONFIG["fp16"],
    gradient_checkpointing=CONFIG["gradient_checkpointing"],

    gradient_checkpointing_kwargs={"use_reentrant": False},

    optim=CONFIG["optim"],
    eval_strategy=CONFIG["eval_strategy"],
    save_strategy=CONFIG["save_strategy"],
    save_total_limit=CONFIG["save_total_limit"],
    load_best_model_at_end=CONFIG["load_best_model_at_end"],
    metric_for_best_model=CONFIG["metric_for_best_model"],
    greater_is_better=CONFIG["greater_is_better"],
    logging_dir=f"{CONFIG['output_dir']}/logs",
    logging_steps=50,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,

    processing_class=tokenizer,

    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG["early_stopping_patience"])],
)

eff_batch = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]
total_steps = (len(train_tok) // eff_batch) * CONFIG["num_train_epochs"]
print(f" Trainer ready")
print(f"   Effective batch size: {eff_batch}")
print(f"   Estimated steps: ~{total_steps:,}")
print(f"   LoRA trainable: {trainable_params:,} params")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


 Trainer ready
   Effective batch size: 32
   Estimated steps: ~13,176
   LoRA trainable: 1,181,954 params


In [ ]:
train_result = trainer.train()

print(f"   Loss: {train_result.training_loss:.4f}")
print(f"   Steps: {train_result.global_step}")
runtime = train_result.metrics.get('train_runtime', 0)
print(f"   Time: {runtime/60:.1f} phút")

Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,1.192336,0.605953,0.445699,0.420753,0.473790,0.795130
2,1.199504,0.594622,0.440589,0.357410,0.574230,0.746504


   Loss: 1.1910
   Steps: 8786
   Time: 52.6 phút


In [ ]:
best_path = os.path.join(CONFIG["output_dir"], "best-lora-adapter")
model.save_pretrained(best_path)
tokenizer.save_pretrained(best_path)

adapter_size = sum(
    os.path.getsize(os.path.join(best_path, f))
    for f in os.listdir(best_path)
    if os.path.isfile(os.path.join(best_path, f))
) / 1e6
print(f" LoRA adapter saved: {best_path}")

 LoRA adapter saved: /content/drive/MyDrive/NLP_Projects/phobert-ext-summ/best-lora-adapter


## 7. Đánh giá

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_tok, metric_key_prefix="test")

print(f"   F1:        {test_metrics.get('test_f1', 0):.4f}")
print(f"   Precision: {test_metrics.get('test_precision', 0):.4f}")
print(f"   Recall:    {test_metrics.get('test_recall', 0):.4f}")
print(f"   Accuracy:  {test_metrics.get('test_accuracy', 0):.4f}")

early stopping required metric_for_best_model, but did not find eval_f1 so early stopping is disabled


   F1:        0.4205
   Precision: 0.4012
   Recall:    0.4417
   Accuracy:  0.7967


In [ ]:
rouge_metric = evaluate.load("rouge")

@torch.no_grad()
def score_sentences(sentences: List[str], model, tokenizer) -> List[float]:
    model.eval()
    if not sentences:
        return []

    encodings = tokenizer(
        sentences,
        max_length=CONFIG["max_length"],
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)

    outputs = model(**encodings)
    probs = torch.softmax(outputs.logits, dim=-1)[:, 1]
    return probs.cpu().tolist()


def extractive_summarize(text: str, model, tokenizer, top_k: int = 3) -> str:
    sents = split_sentences_vi(text)[:CONFIG["max_sentences"]]
    if not sents:
        return ""

    scores = score_sentences(sents, model, tokenizer)
    top_k = min(top_k, len(sents))
    top_indices = sorted(
        sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    )
    return " ".join(sents[i] for i in top_indices)


test_doc_ids = test_df["doc_id"].unique()
predictions, references = [], []

for doc_id in tqdm(test_doc_ids, desc="ROUGE eval"):
    doc = str(raw_dataset[int(doc_id)][CONFIG["text_column"]])
    ref = str(raw_dataset[int(doc_id)][CONFIG["summary_column"]])

    pred = extractive_summarize(doc, model, tokenizer, top_k=CONFIG["top_k_sentences"])
    if pred:
        predictions.append(pred)
        references.append(ref)

rouge_results = rouge_metric.compute(
    predictions=predictions, references=references, use_stemmer=False
)

print(f"   ROUGE-1:    {rouge_results['rouge1']:.4f}")
print(f"   ROUGE-2:    {rouge_results['rouge2']:.4f}")
print(f"   ROUGE-L:    {rouge_results['rougeL']:.4f}")
print(f"   ROUGE-Lsum: {rouge_results['rougeLsum']:.4f}")
print(f"   Documents:  {len(predictions)}")
print(f"{'='*50}")

ROUGE eval:   0%|          | 0/1500 [00:00<?, ?it/s]

   ROUGE-1:    0.3447
   ROUGE-2:    0.1823
   ROUGE-L:    0.2311
   ROUGE-Lsum: 0.2321
   Documents:  1500


In [ ]:
all_results = {
    "sentence_metrics": {k: float(v) for k, v in test_metrics.items()},
    "rouge_metrics": {k: float(v) for k, v in rouge_results.items()},
    "config": {k: str(v) for k, v in CONFIG.items()},
}
with open(os.path.join(CONFIG["output_dir"], "results.json"), "w") as f:
    json.dump(all_results, f, indent=2)
print("Results saved!")

Results saved!


## 8. Demo

In [ ]:
demo_ids = test_df["doc_id"].unique()[:5]

for doc_id in demo_ids:
    doc = str(raw_dataset[int(doc_id)][CONFIG["text_column"]])
    ref = str(raw_dataset[int(doc_id)][CONFIG["summary_column"]])

    sents = split_sentences_vi(doc)[:CONFIG["max_sentences"]]

    if not sents:
        continue

    scores = score_sentences(sents, model, tokenizer)
    top_k = CONFIG["top_k_sentences"]
    top_indices = set(sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k])

    print(f"\n{'─'*70}")
    print(f"Doc #{doc_id} — {len(sents)} câu")
    print(f"{'─'*70}")

    for i, (s, sc) in enumerate(zip(sents, scores)):
        short = s[:90] + "..." if len(s) > 90 else s
        print(f"   [{i:2d}] ({sc:.3f}) {short}")

    pred = " ".join(sents[i] for i in sorted(top_indices))

    print(f"\n Reference:")
    print(f"      {ref[:300]}")
    print(f"\n Prediction:")
    print(f"      {pred[:300]}")

    r = rouge_metric.compute(predictions=[pred], references=[ref])
    print(f"\n   ROUGE-1: {r['rouge1']:.3f} | ROUGE-2: {r['rouge2']:.3f} | ROUGE-L: {r['rougeL']:.3f}")


──────────────────────────────────────────────────────────────────────
Doc #3 — 1 câu
──────────────────────────────────────────────────────────────────────
   [ 0] (0.955) Khoảng 9h30 sáng 5/6 , anh Đào Nhật Tuấn ( 30 tuổi ) lên khu vực hố Thung , thôn Đại La , ...

 Reference:
      Thấy con chim bị mắc bẫy rơi xuống hồ nước , anh Tuấn bơi ra bắt nhưng bị đuối nước .

 Prediction:
      Khoảng 9h30 sáng 5/6 , anh Đào Nhật Tuấn ( 30 tuổi ) lên khu vực hố Thung , thôn Đại La , xã Hoà Sơn , huyện Hoà Vang ( Đà Nẵng ) để giăng lưới bẫy chim trời .Lực lượng cứu nạn tìm thấy thi thể nạn nhân vào chiều cùng ngày .Thấy con chim mắc bẫy rơi xuống hố nước , anh Tuấn bơi ra bắt .Do hố nước qu

   ROUGE-1: 0.238 | ROUGE-2: 0.201 | ROUGE-L: 0.215

──────────────────────────────────────────────────────────────────────
Doc #19 — 1 câu
──────────────────────────────────────────────────────────────────────
   [ 0] (0.966) Ngày 21-10 văn phòng Bộ Y tế soạn một thông báo chính thức , giải thích về đề 

In [ ]:
custom = """
Sáng 15/3, tại Hà Nội, Thủ tướng Chính phủ đã chủ trì Hội nghị trực tuyến toàn quốc về
phát triển công nghệ trí tuệ nhân tạo (AI). Thủ tướng nhấn mạnh Việt Nam cần nắm bắt
cơ hội từ cuộc cách mạng AI để phát triển kinh tế - xã hội, đồng thời xây dựng hành lang
pháp lý phù hợp. Theo báo cáo, thị trường AI tại Việt Nam đang tăng trưởng mạnh với
hơn 500 doanh nghiệp hoạt động trong lĩnh vực này. Bộ Khoa học và Công nghệ đã trình
Chiến lược quốc gia về AI đến năm 2030, trong đó đặt mục tiêu đưa Việt Nam vào nhóm
4 nước dẫn đầu ASEAN về nghiên cứu và ứng dụng AI. Nhiều đại biểu cho rằng cần
đầu tư mạnh mẽ vào đào tạo nhân lực, đặc biệt là thu hút các chuyên gia Việt kiều
trong lĩnh vực AI về nước làm việc.
"""

sents = split_sentences_vi(custom)
scores = score_sentences(sents, model, tokenizer)

print("Custom Input:")
for i, (s, sc) in enumerate(zip(sents, scores)):
    icon = "" if sc > sorted(scores, reverse=True)[min(2, len(scores)-1)] else "  "
    print(f"   {icon} [{i}] ({sc:.3f}) {s}")

pred = extractive_summarize(custom, model, tokenizer, top_k=2)
print(f"\n Summary (top 2):")
print(f"   {pred}")

Custom Input:
    [0] (0.443) Sáng 15/3, tại Hà Nội, Thủ tướng Chính phủ đã chủ trì Hội nghị trực tuyến toàn quốc về
phát triển công nghệ trí tuệ nhân tạo (AI).
      [1] (0.350) Thủ tướng nhấn mạnh Việt Nam cần nắm bắt
cơ hội từ cuộc cách mạng AI để phát triển kinh tế - xã hội, đồng thời xây dựng hành lang
pháp lý phù hợp.
      [2] (0.239) Theo báo cáo, thị trường AI tại Việt Nam đang tăng trưởng mạnh với
hơn 500 doanh nghiệp hoạt động trong lĩnh vực này.
      [3] (0.345) Bộ Khoa học và Công nghệ đã trình
Chiến lược quốc gia về AI đến năm 2030, trong đó đặt mục tiêu đưa Việt Nam vào nhóm
4 nước dẫn đầu ASEAN về nghiên cứu và ứng dụng AI.
    [4] (0.353) Nhiều đại biểu cho rằng cần
đầu tư mạnh mẽ vào đào tạo nhân lực, đặc biệt là thu hút các chuyên gia Việt kiều
trong lĩnh vực AI về nước làm việc.

 Summary (top 2):
   Sáng 15/3, tại Hà Nội, Thủ tướng Chính phủ đã chủ trì Hội nghị trực tuyến toàn quốc về
phát triển công nghệ trí tuệ nhân tạo (AI). Nhiều đại biểu cho rằng cần
đầu tư m